# Leakage-Safe Water Quality Index Forecasting

This notebook forecasts a river station's **next-year Water Quality Index (WQI)** from measurements available in the current year: pH, dissolved oxygen (DO), biochemical oxygen demand (BOD), total and fecal coliform counts, and nitrate. It uses the reusable implementation in `src/model.py`; no modeling logic is duplicated here.

Run this notebook from the repository root after `uv sync --group dev`.

## Research-paper context and project extension

The included paper (`paper/s12145-025-01865-2.pdf`) evaluates individual XGBR, RFR, SVR, and RNN models on 432 **monthly** Haridwar observations from 2017–2022, using an 80:20 split. Its reported R² values are 0.9128 (XGBR), 0.8927 (RFR), 0.9246 (SVR), and 0.9655 (RNN).

This repository contains 2003–2014 nationwide observations rather than that Haridwar dataset. LightGBM and the stacked ensemble are therefore project extensions. Paper scores are context for interpretation, not a target to optimize against.

## Why this is a forecast, not same-row fitting

WQI is calculated from water-quality measurements. Training on a WQI calculated from the *same row's* measurements would leak the answer into the inputs and can produce misleadingly high R². Instead, this notebook pairs a station's measurements at year *t* with its calculated WQI at year *t + 1*. Only consecutive-year station records are used.

Duplicate station/year rows are averaged. The documented five-input WQI approximation applies weights DO 0.31, pH 0.11, BOD 0.19, fecal coliform 0.28, and nitrate 0.11.

In [1]:
from pathlib import Path

import pandas as pd

from src.model import (
    FEATURE_COLUMNS,
    TEST_YEAR,
    TRAIN_END_YEAR,
    VALIDATION_YEAR,
    aggregate_station_years,
    apply_feature_engineering,
    build_forecast_pairs,
    generate_shap_summary,
    load_dataset,
    print_results,
    run_pipeline,
    split_by_target_year,
)

DATA_PATH = Path('data/dataset.csv')

## Prepare station-year forecast pairs

The target year controls every split: target years through 2012 form training, 2013 forms validation, and 2014 is the untouched final test set. Numeric imputation and scaling are fit inside the training flow only.

In [2]:
station_years = aggregate_station_years(load_dataset(DATA_PATH))
forecast_pairs = build_forecast_pairs(apply_feature_engineering(station_years))
partitions = split_by_target_year(forecast_pairs)

pd.DataFrame(
    {
        'partition': partitions.keys(),
        'target-year rule': [
            f'<= {TRAIN_END_YEAR}',
            str(VALIDATION_YEAR),
            str(TEST_YEAR),
        ],
        'forecast pairs': [len(partition) for partition in partitions.values()],
    }
)

,partition,target-year rule,forecast pairs
0,train,<= 2012,707
1,validation,2013,228
2,test,2014,220


## Benchmark models and generalization checks

The benchmark evaluates median and persistence baselines, Random Forest, XGBoost, LightGBM, SVR, and a Ridge meta-model stacked from chronological out-of-fold base-model predictions. The stack never uses random cross-validation or test labels.

Metrics are R², MAE, RMSE, and MSE. A train-to-validation R² drop above 0.10 is reported as an overfitting warning; model choices should be made from validation metrics before reading the final test results.

In [3]:
results = run_pipeline(DATA_PATH)
print_results(results)

metric_table = pd.concat(
    {partition: pd.DataFrame(metrics).T for partition, metrics in results.items()},
    names=['partition', 'model'],
).round(4)
metric_table


Train metrics
  random_forest          R2=0.9430  MAE=2.3871  RMSE=3.3058  MSE=10.9282
  xgboost                R2=0.9646  MAE=1.9797  RMSE=2.6038  MSE=6.7798
  lightgbm               R2=0.9427  MAE=2.4912  RMSE=3.3145  MSE=10.9856
  svr                    R2=0.7959  MAE=4.4903  RMSE=6.2557  MSE=39.1336
  median_baseline        R2=-0.0722  MAE=9.4793  RMSE=14.3371  MSE=205.5518
  persistence_baseline   R2=0.8383  MAE=3.9373  RMSE=5.5681  MSE=31.0041
  stacked_ensemble       R2=0.9315  MAE=2.7483  RMSE=3.6231  MSE=13.1267

Validation metrics
  random_forest          R2=0.8513  MAE=3.8784  RMSE=5.1672  MSE=26.7005
  xgboost                R2=0.8504  MAE=3.7454  RMSE=5.1832  MSE=26.8652
  lightgbm               R2=0.8600  MAE=3.6774  RMSE=5.0135  MSE=25.1351
  svr                    R2=0.8338  MAE=4.0938  RMSE=5.4625  MSE=29.8388
  median_baseline        R2=-0.0786  MAE=9.0126  RMSE=13.9175  MSE=193.6971
  persistence_baseline   R2=0.8790  MAE=3.2872  RMSE=4.6623  MSE=21.7370
  stacked_e

r2     mae     rmse       mse
partition  model                                                  
train      random_forest         0.9430  2.3871   3.3058   10.9282
           xgboost               0.9646  1.9797   2.6038    6.7798
           lightgbm              0.9427  2.4912   3.3145   10.9856
           svr                   0.7959  4.4903   6.2557   39.1336
           median_baseline      -0.0722  9.4793  14.3371  205.5518
           persistence_baseline  0.8383  3.9373   5.5681   31.0041
           stacked_ensemble      0.9315  2.7483   3.6231   13.1267
validation random_forest         0.8513  3.8784   5.1672   26.7005
           xgboost               0.8504  3.7454   5.1832   26.8652
           lightgbm              0.8600  3.6774   5.0135   25.1351
           svr                   0.8338  4.0938   5.4625   29.8388
           median_baseline      -0.0786  9.0126  13.9175  193.6971
           persistence_baseline  0.8790  3.2872   4.6623   21.7370
           stacked_ensemble      0.8551  3.7794   5.1003   26.0130
test       random_forest         0.8648  3.9590   5.2386   27.4426
           xgboost               0.8562  3.9517   5.4021   29.1822
           lightgbm              0.8521  4.0291   5.4795   30.0245
           svr                   0.7517  5.0512   7.0992   50.3988
           median_baseline      -0.0988  9.5149  14.9329  222.9918
           persistence_baseline  0.9076  3.1405   4.3294   18.7440
           stacked_ensemble      0.8554  4.0033   5.4168   29.3414

## SHAP explainability

SHAP explains the fitted Random Forest on held-out test samples. It describes feature influence in this model; it does not establish causation. The output path is explicit so generated artifacts stay outside the repository unless intentionally saved elsewhere.

In [ ]:
shap_output = Path('/tmp/wqi-random-forest-shap.png')
generate_shap_summary(DATA_PATH, shap_output)
shap_output

## RNN: deferred until sequence validation

The research paper's RNN is appropriate for its monthly time series. This data is annual with irregular station histories, so an RNN is intentionally not included in the current benchmark. Implement it only after choosing and testing a sequence length, handling missing years, and ensuring all historical windows precede their targets. Install the optional dependency when needed with `uv sync --group rnn`.